# P11 — Generación aumentada por recuperación para tareas de PLN intensivas en conocimiento

## 1. Título y paper

**Paper:** *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*  
**Autoría:** Patrick Lewis, Ethan Perez, Aleksandra Piktus, Fabio Petroni, y otros  
**Año y venue:** 2020 · arXiv:2005.11401 · NeurIPS 2020  
**Nivel:** L3 · **Motor:** `rag`  
**Ficha completa:** [`P11_rag`](../../papers/foundational/P11_rag/README.md)

**Hito:** Separa el conocimiento (índice consultable y actualizable) del razonamiento (parámetros del modelo).

- [arXiv:2005.11401](https://arxiv.org/abs/2005.11401)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Todo lo que un modelo sabe está congelado en sus pesos: no se puede actualizar sin reentrenar, ni auditar de dónde salió una afirmación.
2. Ejecutar una implementación mínima de la propuesta: Combinar un recuperador denso (DPR) sobre un índice de Wikipedia con un generador seq2seq (BART), entrenados de forma conjunta.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P05
- P10
- Karpukhin et al. (2020), DPR


## 4. Intuición

Un examen a libro cerrado frente a un examen a libro abierto. En el primero, si no lo recuerdas, lo inventas. En el segundo, buscas la página, la citas y quien corrige puede verificarla.


## 5. Concepto mínimo

```text
p(y | x) ≈ Σ_{z ∈ top-k(x)} p_η(z | x) · p_θ(y | x, z)
```

`p_η` es el recuperador (memoria **no paramétrica**, actualizable sin reentrenar) y `p_θ` el generador (memoria **paramétrica**). RAG separa lo que se sabe de cómo se razona.


## 6. Código explicado

El motor recupera por similitud léxica, genera con citas y muestra el contraste con la respuesta sin recuperación.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('rag', seed=7)['result']
print('consulta:', r['query'], '\n')
for fila in r['ranking']:
    print(f"  {fila['doc']} score={fila['score']:.3f} · {fila['text'][:60]}…")
print('\ncon recuperación :', r['respuesta_con_citas'])
print('sin recuperación :', r['respuesta_sin_recuperacion'])

## 7. Predicción antes de ejecutar

1. ¿Qué documento quedará primero: el de la sanción o el de la entrada en vigor?
2. ¿Qué score tendrá el documento sobre hornear pan?
3. Si el recuperador fallara, ¿el generador lo notaría?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
consultas = [
    'cuando entro en vigor la ley de transparencia algoritmica',
    'a que temperatura se hornea el pan',
    'quien gano el mundial de 1986',
]
documentos = {f['doc']: f['text'] for f in r['ranking']}

def tf(texto):
    d = {}
    for t in texto.lower().replace('.', '').split():
        d[t] = d.get(t, 0) + 1
    return d

def coseno(a, b):
    claves = set(a) | set(b)
    va = [a.get(k, 0) for k in claves]
    vb = [b.get(k, 0) for k in claves]
    na = sum(x * x for x in va) ** 0.5
    nb = sum(x * x for x in vb) ** 0.5
    return sum(x * y for x, y in zip(va, vb)) / (na * nb) if na and nb else 0.0

for q in consultas:
    mejor = max(documentos.items(), key=lambda kv: coseno(tf(q), tf(kv[1])))
    print(f'{q[:45]:<47} → {mejor[0]} (score {coseno(tf(q), tf(mejor[1])):.3f})')

## 9. Salida interpretable

La tercera consulta (mundial de 1986) **no tiene respuesta en el corpus** y aun así el recuperador devuelve el documento «menos malo» con un score bajo. Un sistema honesto usa un umbral: por debajo de él, la respuesta correcta es «no lo sé».


## 10. Comentario pedagógico

Recuperar no es responder. Los tres fallos típicos son independientes: (1) el documento correcto no está en el índice, (2) está pero no se recupera, (3) se recupera y el generador lo contradice. Evaluar RAG exige medir los tres por separado.


## 11. Error o anti-patrón deliberado

Anti-patrón: dar por buena una respuesta porque «lleva citas», sin comprobar que la cita sostiene la afirmación.


In [ ]:
respuesta_falsa = 'La sanción máxima es del 12 % de la facturación [d4].'
print(respuesta_falsa)
print('La cita [d4] existe y es relevante… pero el número NO está en d4.')
print('Esto es una alucinación CON cita: la más difícil de detectar a simple vista.')

## 12. Corrección

La verificación mínima: cada afirmación numérica debe aparecer literalmente en el documento citado.


In [ ]:
def verificar(afirmacion_numero, doc_texto):
    return afirmacion_numero in doc_texto

d4 = documentos['d4']
print('d4 =', d4)
print('¿aparece «4 por ciento»? ', verificar('4 por ciento', d4))
print('¿aparece «12 por ciento»?', verificar('12 por ciento', d4))

## 13. Desafío guiado

Añade un umbral de score por debajo del cual el sistema se niega a responder.


In [ ]:
UMBRAL = 0.35
for q in consultas:
    mejor_doc, mejor_txt = max(documentos.items(), key=lambda kv: coseno(tf(q), tf(kv[1])))
    score = coseno(tf(q), tf(mejor_txt))
    if score < UMBRAL:
        print(f'{q[:45]:<47} → ABSTENCIÓN (score {score:.3f} < {UMBRAL})')
    else:
        print(f'{q[:45]:<47} → responder con [{mejor_doc}] (score {score:.3f})')

## 14. Desafío autónomo

Construye un RAG sobre 50 documentos propios. Mide por separado: recall@k del recuperador, fidelidad de la respuesta al contexto y tasa de abstención correcta. Reporta los tres.


## 15. Evidencia de aprendizaje

Guarda el ranking, el caso de alucinación con cita, la verificación literal y el mecanismo de abstención con su umbral.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P11_rag/README.md) · evaluación formal: [`assessments/papers/P11_rag.md`](../../assessments/papers/P11_rag.md)


## 16. Cierre

El modelo ya puede citar. Todavía no está alineado con lo que una persona espera al pedirle algo: eso exige aprender de preferencias humanas.


## 17. Conexión con el siguiente hito

- P13

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
